# `EukaryoticToeholdGate` — usage example (trailing-Kozak layout)

A real, end-to-end run of the single-input eukaryotic toehold switch —
`EukaryoticToeholdGate` in `engine.gates.toehold`. This class is fully implemented
and tested (`tests/engine/gates/test_toehold.py`, 45 passing tests as of this
writing). This is a sibling to [`toehold.ipynb`](toehold.ipynb) (which drives the
gate through a stub `FoldEngine` for fast, dependency-free iteration and covers both
hosts generically) — here we build `EukaryoticToeholdGate` specifically and use the
**real** `FoldEngine` (ViennaRNA) throughout, so every number below is a genuine
fold, not a placeholder.

`ToeholdGate` builds two structurally different eukaryotic layouts (commits
`d754812`, `ee2f5f6`):

* **`"loop"`** — Kozak embedded in the hairpin loop, ported unmodified from the
  prokaryotic mechanism (steric occlusion of the start codon). Unvalidated for a
  eukaryotic toehold specifically — see `KOZAK_LAYOUTS`'s docstring. It also turns
  out Kozak and AUG are *not* adjacent in this layout (a 6 nt stem-closing segment
  sits between them), which breaks the Kozak consensus's own adjacency requirement.
* **`"trailing"`** — Kozak and the start codon sit *after* the closed hairpin
  instead, modelling scanning-ribosome blockage (docs/modalities.md) rather than
  direct start-codon occlusion. This is the layout the team's own eukaryotic
  scripts actually build (`plasmid_prefix + trg_bind_region + loop + stem_down +
  kozak` — Kozak last), and Kozak is genuinely adjacent to AUG here. It also carries
  no trailing `LINKER_SEQUENCE` — cap-dependent scanning initiates the instant the
  40S subunit meets Kozak+AUG, so nothing after the start codon matters to finding
  it, and the payload attaches directly.

It also optionally takes the real effector gene (`payload`, commit `74bf7b4`) and
folds its own first nucleotides into every design instead of a placeholder — because
the real downstream sequence can change which design actually scores best, not just
which one looks best in isolation.

This notebook builds the gate restricted to **`"trailing"`** only
(`kozak_layouts=("trailing",)`), pools designs across the **top 150 trigger
candidates** (not just the single best-scoring one), and reserves at least 40 of
those 150 from the 3&#8242; UTR (from 100 nt before the CDS ends to the end of the
transcript) so that region gets a real chance to compete rather than being crowded
out by however TriggerScorer's global ranking happens to fall.

No Django, no worker, no pipeline — just the gate class, constructed and called
directly, the way `pipeline.py` would use it internally.

## Setup

In [ ]:
# Put <repo>/src on the path. Search upward from cwd for pyproject.toml so this works
# wherever Jupyter is launched from.
import sys
from pathlib import Path

for _base in (Path.cwd(), *Path.cwd().parents):
    if (_base / "pyproject.toml").exists():
        _src = _base / "src"
        if str(_src) not in sys.path:
            sys.path.insert(0, str(_src))
        break

from engine.domain import Host, Regulation, SelectedGene, TriggerSet, Constraints
from engine.gates.toehold import EukaryoticToeholdGate
from engine.gates.tools.folding import FoldEngine
from engine.gates.tools.translation import TranslationScorer
from engine.gates.tools.codons import CodonOptimizer
from engine.stages.folding import FoldProfiler
from engine.stages.motifs import MotifScreener
from engine.stages.off_target import OffTargetScanner
from engine.stages.triggers import TriggerScorer
from engine import sequences as sq

## 1. Build the tools, once

Per `CLAUDE.md` §5: tools are constructed once and handed to the gate, never built
inside a stage or family. `FoldEngine`'s cache is only useful if every caller shares
one instance — a second `FoldEngine()` means a cold cache and, worse, a second
chance to fold at a different temperature.

In [ ]:
host = Host.HUMAN  # HUMAN | YEAST both take the eukaryotic (Kozak) track

folder = FoldEngine(temperature=37.0)
translation = TranslationScorer(host)
codons = CodonOptimizer(host)

# The real effector gene — from `eff` in CERNAL_FUNCTIONS.py (a GFP-family CDS, already
# RNA, 759 nt, tandem stop UAA UAA). When set, the gate folds its real first
# PAYLOAD_HEAD_LENGTH nt into every design instead of a placeholder — see
# PAYLOAD_HEAD_LENGTH's docstring (commit 74bf7b4) for why this can change which
# design scores best. Set to None to see the placeholder behaviour instead.
PAYLOAD_CDS = (
    "AUGCGUAAAGGAGAAGAACUUUUCACUGGAGUUGUCCCAAUUCUUGUUGAAUUAGAUGGUGAUGUUAAUGGGCACAAAUUUUCUGUCAG"
    "UGGAGAGGGUGAAGGUGAUGCAACAUACGGAAAACUUACCCUUAAAUUUAUUUGCACUACUGGAAAACUACCUGUUCCGUGGCCAACAC"
    "UUGUCACUACUUUCGGUUAUGGUGUUCAAUGCUUUGCGAGAUACCCAGAUCACAUGAAACAGCAUGACUUUUUCAAGAGUGCCAUGCCC"
    "GAAGGUUACGUACAGGAAAGAACUAUAUUUUUCAAAGAUGACGGGAACUACAAGACACGUGCUGAAGUCAAGUUUGAAGGUGAUACCCU"
    "UGUUAAUAGAAUCGAGUUAAAAGGUAUUGAUUUUAAAGAAGAUGGAAACAUUCUUGGACACAAAUUGGAAUACAACUAUAACUCACACA"
    "AUGUAUACAUCAUGGCAGACAAACAAAAGAAUGGAAUCAAAGUUAACUUCAAAAUUAGACACAACAUUGAAGAUGGAAGCGUUCAACUA"
    "GCAGACCAUUAUCAACAAAAUACUCCGAUUGGCGAUGGCCCUGUCCUUUUACCAGACAACCAUUACCUGUCCACACAAUCUGCCCUUUC"
    "GAAAGAUCCCAACGAAAAGAGAGACCACAUGGUCCUUCUUGAGUUUGUAACCGCUGCUGGGAUUACACAUGGCAUGGAUGAACUAUACA"
    "AAAGGCCUGCAGCAAACGACGAAAACUACGCUGCAUCAGUUUAAUAA"
)

# kozak_layouts restricts generate_designs to the "trailing" layout only — the default
# (omit this argument) sweeps both "loop" and "trailing" and lets engine.scoring rank
# across them. This mirrors how `host` is a constructor parameter rather than a
# subclass (docs/engine.md §2.4).
gate = EukaryoticToeholdGate(
    host, folder, translation, codons, kozak_layouts=("trailing",), payload=PAYLOAD_CDS
)
print(gate.required_tools())
print("kozak_layouts:", gate.kozak_layouts)
print("payload_head:", gate.payload_head)

## 2. Pick trigger candidates — via the real `TriggerScorer` (stage 2)

`TriggerScorer.score` is fully implemented (`tests/engine/test_triggers.py`, 47
passing tests): it slides every window of every length in
`constraints.trigger_lengths` across the transcript, screens out forbidden motifs,
folds each survivor for `openness`/`accessibility`/`mfe` via the same `FoldEngine`
the gate uses, checks off-targets, ranks by `accessibility * segment_specificity`,
and yields the top candidates per gene — so we use it for real here instead of
hand-picking one window.

`OffTargetScanner` itself is still a stub (`find_similar`/`scan_trigger` raise
`NotImplementedError`) — **except** when handed an empty transcriptome, which is a
deliberate early-return for exactly this case (a `direct` submission, or a demo like
this one, with no reference index to scan against): `scan_trigger` returns a clean
`OffTargetReport(hits=(), penalty=0.0)` rather than raising. That is a real,
documented behaviour of the class, not a workaround.

Earlier versions of this notebook carried only `candidates[0]`, then the top 5,
through — which never gave a weaker-looking trigger the chance to turn out to fold
into a better switch. This version carries the top 150 through instead, with a floor
on how many must come from the 3&#8242; UTR rather than leaving that to chance — see the
cell below.

In [ ]:
# The full-length human AREG mRNA, as cDNA/DNA notation (T, not U) — same alphabet trap
# CLAUDE.md warns about, so normalise with to_rna() before anything else touches it.
transcript_dna = (
    "AGACGTTCGCACACCTGGGTGCCAGCGCCCCAGAGGTCCCGGGACAGCCCGAGGCGCCGCGCCCGCCGCCCCGAGCTCCCC"
    "AAGCCTTCGAGAGCGGCGCACACTCCCGGTCTCCACTCGCTCTTCCAACACCCGCTCGTTTTGGCGGCAGCTCGTGTCCCA"
    "GAGACCGAGTTGCCCCAGAGACCGAGACGCCGCCGCTGCGAAGGACCAATGAGAGCCCCGCTGCTACCGCCGGCGCCGGTG"
    "GTGCTGTCGCTCTTGATACTCGGCTCAGGCCATTATGCTGCTGGATTGGACCTCAATGACACCTACTCTGGGAAGCGTGAA"
    "CCATTTTCTGGGGACCACAGTGCTGATGGATTTGAGGTTACCTCAAGAAGTGAGATGTCTTCAGGGAGTGAGATTTCCCCT"
    "GTGAGTGAAATGCCTTCTAGTAGTGAACCGTCCTCGGGAGCCGACTATGACTACTCAGAAGAGTATGATAACGAACCACAA"
    "ATACCTGGCTATATTGTCGATGATTCAGTCAGAGTTGAACAGGTAGTTAAGCCCCCCCAAAACAAGACGGAAAGTGAAAAT"
    "ACTTCAGATAAACCCAAAAGAAAGAAAAAGGGAGGCAAAAATGGAAAAAATAGAAGAAACAGAAAGAAGAAAAATCCATGT"
    "AATGCAGAATTTCAAAATTTCTGCATTCACGGAGAATGCAAATATATAGAGCACCTGGAAGCAGTAACATGCAAATGTCA"
    "GCAAGAATATTTCGGTGAACGGTGTGGGGAAAAGTCCATGAAAACTCACAGCATGATTGACAGTAGTTTATCAAAAATTG"
    "CATTAGCAGCCATAGCTGCCTTTATGTCTGCTGTGATCCTCACAGCTGTTGCTGTTATTACAGTCCAGCTTAGAAGACAA"
    "TACGTCAGGAAATATGAAGGAGAAGCTGAGGAACGAAAGAAACTTCGACAAGAGAATGGAAATGTACATGCTATAGCATA"
    "ACTGAAGATAAAATTACAGGATATCACATTGGAGTCACTGCCAAGTCATAGCCATAAATGATGAGTCGGTCCTCTTTCCA"
    "GTGGATCATAAGACAATGGACCCTTTTTGTTATGATGGTTTTAAACTTTCAATTGTCACTTTTTATGCTATTTCTGTATA"
    "TAAAGGTGCACGAAGGTAAAAAGTATTTTTTCAAGTTGTAAATAATTTATTTAATATTTAATGGAAGTGTATTTATTTTA"
    "CAGCTCATTAAACTTTTTTAACCAAA"
)
transcript = sq.to_rna(transcript_dna)
assert sq.is_valid_rna(transcript)
print(f"transcript length: {len(transcript)} nt")

## Locate the CDS, to define the 3&#8242; UTR

Nothing in this engine parses gene annotation (no GTF/GFF loader exists — the CSV/
sequence-database parser is one of the two things `CLAUDE.md` §7 names as having "no
home yet"). To split this transcript into CDS and 3&#8242; UTR at all, approximate the
CDS as the **longest open reading frame** — every AUG in every frame, paired with its
nearest downstream in-frame stop, keeping the longest. This is a heuristic standing in
for real annotation, not something to trust for a transcript whose true CDS isn't
already known by other means — flagged, not silently treated as ground truth.

In [ ]:
def longest_orf(sequence):
    """(start, stop_codon_start, length) of the longest AUG-to-in-frame-stop ORF,
    across all three frames. A heuristic CDS finder, not a gene-model parser."""
    best = None
    for frame in (0, 1, 2):
        augs = sq.find_augs(sequence, frame=frame)
        stops = sq.find_stops(sequence, frame=frame)
        for start in augs:
            downstream_stops = [s for s in stops if s > start]
            if not downstream_stops:
                continue
            stop = min(downstream_stops)
            length = stop - start
            if best is None or length > best[2]:
                best = (start, stop, length, frame)
    return best


orf_start, stop_codon_start, orf_length, orf_frame = longest_orf(transcript)
cds_end = stop_codon_start + 3  # exclusive, i.e. transcript[cds_end:] is the 3' UTR

print(f"longest ORF: {orf_start}-{stop_codon_start} ({orf_length} nt, frame {orf_frame})")
print(f"cds_end: {cds_end}  (transcript length {len(transcript)}, "
      f"3' UTR is {len(transcript) - cds_end} nt)")

In [ ]:
# Stage-2 tools, built once (same injection rule as the gate's own tools).
profiler = FoldProfiler()
screener = MotifScreener()
off_target = OffTargetScanner(transcriptome={})  # empty: no reference index for this demo
scorer = TriggerScorer(profiler, off_target, screener, folder)  # shares the gate's FoldEngine

# TriggerScorer.TOP_K_PER_GENE defaults to 50 — a search-budget knob (its own
# docstring), not a ceiling meaningful here. Raised so the 3' UTR quota below has the
# whole ranked pool to draw from, not just whatever survived a 50-candidate cutoff.
scorer.TOP_K_PER_GENE = 5000

# In a real run this comes from GeneSelector (stage 1); stand in with a minimal
# SelectedGene since this demo starts from a single known transcript.
gene = SelectedGene(
    gene_id="AREG",
    symbol="AREG",
    regulation=Regulation.UP,
    log2_fold_change=2.0,
    score=1.0,
)
constraints = Constraints(trigger_lengths=(30, 36), max_switch_length=200)

candidates = list(scorer.score([gene], {"AREG": transcript}, constraints))
print(f"{len(candidates)} candidate(s) survived screening, best-scoring first\n")
for c in candidates[:5]:
    print(
        f"  {c.trigger_id:22} start={c.start_index:4} len={c.length:2}  "
        f"score={c.score:.3f}  accessibility={c.accessibility:.3f}  gc={c.gc_content:.1f}"
    )

# Select the top N_TRIGGERS, with a floor on how many come from the 3' UTR — a window
# starting anywhere from 100 nt before the CDS ends onward. Nothing in TriggerScorer
# enforces a region quota (it ranks globally by score alone), so this is done here,
# not in the engine: reserve MIN_FROM_UTR slots from the UTR-only pool, then fill the
# rest with whatever scores best overall (UTR leftovers included, so a strong UTR
# candidate isn't capped out of the general competition).
N_TRIGGERS = 150
MIN_FROM_UTR = 40
utr_start = cds_end - 100

utr_pool = [c for c in candidates if c.start_index >= utr_start]
orf_pool = [c for c in candidates if c.start_index < utr_start]
print(f"\n{len(utr_pool)} candidate(s) start in the 3' UTR (>= {utr_start}), "
      f"{len(orf_pool)} start in or before the CDS")

utr_reserved = utr_pool[:MIN_FROM_UTR]
remaining = sorted(utr_pool[MIN_FROM_UTR:] + orf_pool, key=lambda c: c.score, reverse=True)
top_triggers = sorted(
    utr_reserved + remaining[: N_TRIGGERS - len(utr_reserved)],
    key=lambda c: c.score,
    reverse=True,
)

n_utr_selected = sum(1 for c in top_triggers if c.start_index >= utr_start)
print(f"\nselected {len(top_triggers)} trigger(s): {n_utr_selected} from the 3' UTR "
      f"(floor was {MIN_FROM_UTR}), {len(top_triggers) - n_utr_selected} from the CDS")

## 3. Wrap each trigger in its own `TriggerSet`

`TriggerSet` is the circuit's inputs (one activator each here — every trigger builds
its own single-input switch, independently). `Constraints` were already built above,
since `TriggerScorer` needed them too — a run builds `Constraints` once from
`params["constraints"]` and threads the same object through every stage.

In [ ]:
trigger_sets = [TriggerSet(activators=(t,)) for t in top_triggers]

print(f"{len(trigger_sets)} TriggerSet(s) built, e.g.:")
for ts in trigger_sets[:3]:
    print(" ", ts.activators[0].trigger_id, "| arity:", ts.arity, "| logic:", ts.logic_type)

## 4. `is_compatible()` — cheap check before generating anything

Arity, host, trigger length window — nothing here folds. Checked per trigger set,
same as a real run would (a family is checked against every trigger set it might
build from).

In [ ]:
compatible_trigger_sets = []
incompatible = []
for ts in trigger_sets:
    compatibility = gate.is_compatible(ts, constraints)
    if compatibility.ok:
        compatible_trigger_sets.append(ts)
    else:
        incompatible.append((ts, compatibility))

print(f"{len(compatible_trigger_sets)}/{len(trigger_sets)} trigger set(s) compatible")
for ts, reason in incompatible[:5]:
    print(" incompatible:", ts.activators[0].trigger_id, reason)

assert compatible_trigger_sets, "no trigger set survived is_compatible"

## 5. `generate_designs()` — the full candidate pool

Called **once per compatible trigger set**, not just the top one — exactly how a real
run explores the search space, since a lower-ranked trigger can still fold into a
better switch. With `kozak_layouts=("trailing",)`, each trigger set yields one
`GateDesign` per `toehold_lengths` x `TRAILING_LOOP_LENGTHS` x `KOZAK_LINKER_LENGTHS`
combination it supports. Pooled together, this is the candidate set `engine.scoring`
would actually rank in a real run.

In [ ]:
designs = []
for ts in compatible_trigger_sets:
    designs.extend(gate.generate_designs(ts, constraints))

n_from_utr = sum(
    1 for d in designs if d.trigger_set.activators[0].start_index >= utr_start
)
print(f"{len(designs)} design(s) across {len(compatible_trigger_sets)} trigger(s)")
print(f"  {n_from_utr} design(s) trace back to a 3' UTR trigger")
print(f"  {len(designs) - n_from_utr} design(s) trace back to a CDS-region trigger")
print("\nfirst few:")
for d in designs[:5]:
    print(
        f"  {d.design_id:52} {d.length:3} nt  "
        f"trigger={d.trigger_set.activators[0].trigger_id:18}  "
        f"loop_len={d.architecture['loop_len']:2}  "
        f"kozak_linker_len={d.architecture['kozak_linker_len']}"
    )

## 6. `evaluate_design()` — raw metrics and sequences, across the whole pool

**Raw** values only — no normalising, weighting or ranking here, that is
`engine.scoring`'s job. Keys are exactly the metric names `DEFAULT_V1` declares.

For the `"trailing"` layout specifically, `predicted_leakage`/`dynamic_range` are read
from the **toehold+stem region**, not the AUG — the AUG sits outside the hairpin here
and stays roughly accessible whether or not the trigger is bound, so AUG-region
accessibility would not discriminate ON from OFF for this layout (see
`evaluate_design`'s docstring). This is a new, unreviewed proxy — not a port of
anything previously validated.

Because `PAYLOAD_CDS` is set (cell 4), every design below is folded **with the real
effector gene's own first `PAYLOAD_HEAD_LENGTH` nucleotides**, not a placeholder — the
molecule `gate_folding_energy`/`predicted_leakage`/`dynamic_range` are measured on is
the one a ribosome would actually encounter once this gene is attached. Set
`PAYLOAD_CDS = None` in cell 4 and re-run to see how the numbers shift for the exact
same designs without that information.

`ranked` below holds **every** design in the pool (hundreds, from 150 triggers), best
first by `dynamic_range` — the cell only *prints* the top `TOP_DISPLAY`, but nothing
downstream (the report, cell 7's follow-ups) is limited to that display slice.

In [ ]:
all_metrics = [(d, gate.evaluate_design(d)) for d in designs]
ranked = sorted(all_metrics, key=lambda pair: pair[1]["dynamic_range"], reverse=True)

TOP_DISPLAY = 20
print(f"top {TOP_DISPLAY} of {len(ranked)}:\n")
print(
    f"{'trigger':>18}  {'region':>6}  {'loop_len':>8}  {'linker_len':>10}  {'leakage':>8}  "
    f"{'dyn_range':>9}  {'folding_energy':>14}"
)
for d, m in ranked[:TOP_DISPLAY]:
    region = "utr" if d.trigger_set.activators[0].start_index >= utr_start else "cds"
    print(
        f"{d.trigger_set.activators[0].trigger_id:>18}  {region:>6}  "
        f"{d.architecture['loop_len']:>8}  {d.architecture['kozak_linker_len']:>10}  "
        f"{m['predicted_leakage']:>8.3f}  {m['dynamic_range']:>9.3f}  "
        f"{m['gate_folding_energy']:>14.1f}"
    )

print(f"\nsequences, same order, top {TOP_DISPLAY} (best first):\n")
for d, _m in ranked[:TOP_DISPLAY]:
    print(f"{d.design_id}  ({d.length} nt)")
    print(f"  {gate.emit_sequence(d)}")

# Not this gate's job to rank in a real run (engine.scoring owns that) — but picking the
# top-ranked one here to carry through the rest of this notebook's cells.
design, metrics = ranked[0]
print(f"\nbest by dynamic_range: {design.design_id}")
for name, value in metrics.items():
    print(f"  {name:22} {value}")

Three things worth noticing:

1. **A design finally clears `dynamic_range` 1.0.** The best of the 904,
   `eukaryotic_toehold-trig-AREG-563-30-12-trailing-8-0`, scores 1.548 with
   `predicted_leakage` of just 0.018 — a real, working switch by this proxy, not
   another below-1.0 result to explain away. Widening the search from 5 triggers to
   150 is what found it; it wasn't visible in the earlier, smaller pool.
2. **The best design came from the CDS region, not the 3&#8242; UTR**, and every one of
   the top 20 does too — the first 3&#8242; UTR-derived design lands at **rank 78** of
   904 (`dynamic_range` 0.109). The 41-candidate UTR floor (cell 7) guarantees 3&#8242;
   UTR triggers get a fair *chance* to compete; it doesn't guarantee they win, and for
   this transcript and mechanism they mostly don't. That's a real result about *AREG*
   specifically, not a flaw in the quota.
3. **Pool composition**: 904 designs total, 300 tracing back to a 3&#8242; UTR trigger
   and 604 to a CDS-region one (cell 13) — roughly proportional to the 41/109 trigger
   split, since every compatible trigger yields the same 4 designs (`loop_len` x
   `kozak_linker_len` for `toehold_length=12`; the 36 nt triggers also unlock
   `toehold_length` 15 in some cases, mattering little here).

## 7. `emit_sequence()` — the synthesis-ready sequence

In [ ]:
print(gate.emit_sequence(design))

The Kozak element and start codon aren't visually obvious in that raw string — for this
`"trailing"` layout they sit right after the closed hairpin, not inside it (contrast
with `"loop"`, where they'd be buried in the middle). Note also what comes right after
the AUG here: the real effector gene's own first `PAYLOAD_HEAD_LENGTH` nucleotides
(`eff` from `CERNAL_FUNCTIONS.py`, set as `PAYLOAD_CDS` in cell 4), not a placeholder
— `"trailing"` carries no trailing linker of its own (commit `ee2f5f6`); cap-dependent
scanning initiates the instant the 40S subunit meets Kozak+AUG, so nothing after the
start codon plays any role in finding it, and whatever comes after is either the
payload (when known, as folded in here) or nothing at all (commit `74bf7b4`). Locate
Kozak and the start codon explicitly using `design.architecture` (which records
`aug_index` exactly) and the gate's own `KOZAK_EUKARYOTIC` constant:

In [ ]:
seq = design.sequence
aug_index = design.architecture["aug_index"]
kozak_index = seq.find(gate.KOZAK_EUKARYOTIC)
assert kozak_index != -1, "Kozak element not found — architecture assumptions above are stale"

marks = [" "] * len(seq)
for i in range(kozak_index, kozak_index + len(gate.KOZAK_EUKARYOTIC)):
    marks[i] = "K"
for i in range(aug_index, aug_index + 3):
    marks[i] = "A"

print(seq)
print("".join(marks), " K = Kozak (GCCACC)   A = start codon (AUG)")

## 8. Report — top 15, with a 3&#8242; UTR floor and 2D structure

`ReportBuilder`/`StructureRenderer` (`src/engine/stages/reporting.py`, stage 6/S14)
are still stubs — `render_structure`, `render_circuit` and `build` all raise
`NotImplementedError("Step 5")`. This section is **not** that; it's a report built
here in the notebook from the real `ranked` pool computed above, not a pipeline
capability.

`ranked` is already best-first by `dynamic_range`. Taking the top 15 outright could
easily land all 15 in the CDS region (it very nearly does — see cell 18's finding
that the first 3&#8242; UTR design sits at rank 78/904), so the same reserved-floor
approach from trigger selection (cell 7) applies again here: keep the top 15,
but if fewer than `MIN_UTR_IN_REPORT` of them are 3&#8242; UTR-derived, swap in the
best-ranked 3&#8242; UTR designs not already present, evicting the *worst*-ranked
non-UTR entries to make room — never touching rank #1.

In [ ]:
def select_with_region_floor(ranked_pool, n, min_from_region, in_region):
    """Top `n` of `ranked_pool` (already best-first), with at least `min_from_region`
    satisfying `in_region`. If the natural top `n` doesn't have enough, swap in the
    best-ranked region matches not already present, evicting the worst-ranked
    non-matching entries — never the entries that were already in-region."""
    top = list(ranked_pool[:n])
    have = sum(1 for d, m in top if in_region(d))
    if have >= min_from_region:
        return top

    top_ids = {d.design_id for d, m in top}
    region_reserve = [
        pair for pair in ranked_pool if in_region(pair[0]) and pair[0].design_id not in top_ids
    ]
    non_region_in_top = [pair for pair in top if not in_region(pair[0])]

    for extra in region_reserve[: min_from_region - have]:
        if not non_region_in_top:
            break
        evict = non_region_in_top.pop()  # worst-ranked non-region entry (list end)
        top.remove(evict)
        top.append(extra)

    top.sort(key=lambda pair: pair[1]["dynamic_range"], reverse=True)
    return top


def in_3utr(design):
    return design.trigger_set.activators[0].start_index >= utr_start


MIN_UTR_IN_REPORT = 3
REPORT_SIZE = 15
report = select_with_region_floor(ranked, REPORT_SIZE, MIN_UTR_IN_REPORT, in_3utr)

n_utr_in_report = sum(1 for d, m in report if in_3utr(d))
print(f"report: {len(report)} design(s), {n_utr_in_report} from the 3' UTR "
      f"(floor was {MIN_UTR_IN_REPORT})\n")

# Ensemble free energy (FoldEngine.partition — the partition-function total over
# every structure, not just the most stable one) rather than evaluate_design's own
# gate_folding_energy (MFE): "MFE reports a single structure, but RNA in solution
# occupies a distribution" (FoldEngine.partition's own docstring). This doesn't
# change the engine's scored metric anywhere — it's a display choice for this report
# table only.
print(
    f"{'#':>3}  {'trigger':>18}  {'region':>6}  {'geometry (nt)':>14}  "
    f"{'leakage':>8}  {'dyn_range':>9}  {'ensemble ΔG':>12}  {'gc%':>6}"
)
for rank, (d, m) in enumerate(report, 1):
    region = "utr" if in_3utr(d) else "cds"
    arch = d.architecture
    geometry = f"{arch['toehold_length']}/{arch['loop_len']}/{arch['kozak_linker_len']}"
    ensemble_energy = gate.folder.partition(d.sequence)
    print(
        f"{rank:>3}  {d.trigger_set.activators[0].trigger_id:>18}  {region:>6}  "
        f"{geometry:>14}  {m['predicted_leakage']:>8.3f}  {m['dynamic_range']:>9.3f}  "
        f"{ensemble_energy:>12.1f}  {m['gc_content']:>6.1f}"
    )

### 2D structure

`RNA.get_xy_coordinates` (ViennaRNA's own layout algorithm, not a house rule
violation — this is display-only, not a scientific computation, and this notebook
isn't `src/engine/`, so the "only two modules import RNA" rule (`CLAUDE.md` §5)
doesn't apply here) turns a dot-bracket string into 2D coordinates. Each base is
coloured by identity; Kozak and the start codon are outlined, and the 5&#8242;/3&#8242;
ends are labelled, so the mechanism this whole layout exists for is visible at a
glance.

**Tried and reverted: the ensemble's centroid structure (`fc.centroid()`).** A
centroid fold is the more principled choice for "the" ensemble-representative
structure — but calling it in a loop of 15, in the same process as the 904-design
pipeline above, segfaults **reliably** (isolated by bisection: `.mfe()` alone in a
loop here is fine, `.pf()` alone in a loop is fine, `.centroid()` alone in a fresh
process is fine — only `.centroid()` *after* the heavy pipeline, in this process,
crashes). That's a real ViennaRNA Python-binding memory instability under heavy
cumulative `fold_compound` churn, not a bug in this notebook's logic — but not
something to ship a notebook a teammate might run into. **This cell therefore plots
`gate.folder.mfe(seq).structure` instead** — the single lowest-energy structure,
via the same injected, cached `FoldEngine` already used everywhere else, proven
stable at this scale. It's a real predicted fold, still not `design.dot_bracket`
(the generator's idealized target, identical in shape for every design of matching
geometry regardless of how well it actually folds) — just not the ensemble centroid
specifically. `ensemble ΔG` in the table above (cell 7) still comes from `.pf()`
(via `FoldEngine.partition`), which is stable at this scale and unaffected by this.

The layout is computed for the **switch alone**, up to and including the start
codon — the attached payload head is real for every metric elsewhere in this
notebook, but a long unpaired tail made Kozak/AUG curl back toward the middle of the
picture instead of sitting at the 3&#8242; end where they belong, so it's dropped here
for the diagram specifically.

In [ ]:
import RNA
import matplotlib.pyplot as plt

BASE_COLORS = {"A": "#FFD6A5", "U": "#A0C4FF", "G": "#FFADAD", "C": "#CAFFBF"}


def plot_structure_mini(ax, design, label):
    aug_index = design.architecture["aug_index"]
    kozak_index = design.sequence.find(gate.KOZAK_EUKARYOTIC)

    # Switch only, up to and including the start codon — see the markdown above for
    # why (Kozak/AUG land at the 3' end instead of curling into the middle) and why
    # this uses gate.folder.mfe() rather than a raw fc.centroid() call (a real,
    # reproducible ViennaRNA segfault at this scale, isolated and documented above).
    seq = design.sequence[: aug_index + 3]
    structure = gate.folder.mfe(seq).structure

    coords = RNA.get_xy_coordinates(structure)
    xs = [coords.get(i).X for i in range(len(structure))]
    ys = [coords.get(i).Y for i in range(len(structure))]

    ax.plot(xs, ys, "-", color="#B9C2BC", linewidth=1.1, zorder=1)

    stack = []
    for i, c in enumerate(structure):
        if c == "(":
            stack.append(i)
        elif c == ")":
            j = stack.pop()
            ax.plot(
                [xs[i], xs[j]], [ys[i], ys[j]], "-", color="#8899AA",
                linewidth=0.9, zorder=1, alpha=0.7,
            )

    highlight = set(range(kozak_index, kozak_index + len(gate.KOZAK_EUKARYOTIC)))
    highlight |= set(range(aug_index, aug_index + 3))

    for i, base in enumerate(seq):
        is_hl = i in highlight
        ax.scatter(
            xs[i], ys[i],
            s=26 if is_hl else 14,
            color=BASE_COLORS.get(base, "#EEEEEE"),
            edgecolors="#1A2620" if is_hl else "#666666",
            linewidths=1.1 if is_hl else 0.3,
            zorder=3 if is_hl else 2,
        )

    ax.text(xs[0], ys[0] + 9, "5'", fontsize=7, ha="center", color="#345", zorder=4)
    ax.text(xs[-1], ys[-1] + 9, "3'", fontsize=7, ha="center", color="#345", zorder=4)

    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(label, fontsize=9, fontweight="bold", loc="left")


fig, axes = plt.subplots(3, 5, figsize=(18, 11))
for i, (d, _m) in enumerate(report):
    region = "UTR" if in_3utr(d) else "CDS"
    label = f"#{i + 1}  {d.trigger_set.activators[0].trigger_id}  ({region})"
    plot_structure_mini(axes[i // 5][i % 5], d, label)

fig.suptitle(
    "Top 15 trailing-Kozak toeholds against AREG — MFE fold, switch only "
    "(payload head omitted), colour = base identity, outlined = Kozak/AUG",
    fontsize=12,
)
plt.tight_layout()
plt.show()

Base colours: A `#FFD6A5` · U `#A0C4FF` · G `#FFADAD` · C `#CAFFBF` (matching the
convention already used in `CERNAL_FUNCTIONS.py`'s own `plot_rna_structure`).

The correlation with the metrics table is visible directly in these diagrams: #1
(`dynamic_range` 1.55) folds into one clean extended hairpin with Kozak/AUG exposed
in a small accessible loop of their own. The three 3&#8242; UTR designs pulled in to meet
`MIN_UTR_IN_REPORT` land at #13&#8211;#15 — the worst `dynamic_range` in this 15 — and
their folds show why: Kozak/AUG buried inside a branched, multi-junction structure
rather than sitting in a clean loop, visibly less reachable, which is exactly what a
low `dynamic_range` means for this proxy. This is the real predicted fold explaining
the ranking, not the ranking asserted and then illustrated after the fact.

## 9. Publish-ready web report

The cells below generate the same report as a standalone HTML page — matching the
design of the one already published from this data — and write it to
`var/reports/areg_toehold_report.html` (`var/` is this repo's documented location
for runtime output, already gitignored — see `.gitignore`'s own comment on it).

**Running these cells does not publish anything.** Writing the file is something
this notebook's own Python can do; putting it on a live, shareable URL requires
Claude's Artifact tool, which isn't callable from notebook code — that step still
needs a person (or Claude, outside this notebook) to hand the written file to that
tool. What's below gets you from "the data in `report`" to "a finished HTML file
ready to hand off," reproducibly, instead of a one-off untracked script.

In [ ]:
import html as _html


def make_svg(design, width=260, height=210, pad=26):
    """Inline SVG 2D structure for one design — switch only (payload head dropped),
    MFE fold via the same injected FoldEngine used everywhere else in this notebook
    (see section 8's note on why not fc.centroid() — a real, reproducible ViennaRNA
    segfault at this scale). Marks Kozak, the start codon, and the trigger-binding
    site (the toehold itself — the region complementary to the trigger, per
    ToeholdGate.generate_designs: the first toehold_length nt after the leader)."""
    arch = design.architecture
    aug_index = arch["aug_index"]
    kozak_index = design.sequence.find(gate.KOZAK_EUKARYOTIC)
    binding_start = arch["leader_len"]
    binding_end = binding_start + arch["toehold_length"]
    seq = design.sequence[: aug_index + 3]
    structure = gate.folder.mfe(seq).structure

    coords = RNA.get_xy_coordinates(structure)
    xs = [coords.get(i).X for i in range(len(structure))]
    ys = [coords.get(i).Y for i in range(len(structure))]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    scale = min(
        (width - 2 * pad) / max(max_x - min_x, 1), (height - 2 * pad) / max(max_y - min_y, 1)
    )
    px = [pad + (x - min_x) * scale for x in xs]
    py = [height - pad - (y - min_y) * scale for y in ys]

    parts = [
        f'<svg viewBox="0 0 {width} {height}" class="structure-svg" role="img" '
        'aria-label="Predicted 2D structure">'
    ]
    backbone_pts = " ".join(f"{px[i]:.1f},{py[i]:.1f}" for i in range(len(px)))
    parts.append(
        f'<polyline points="{backbone_pts}" fill="none" stroke="var(--struct-backbone)" '
        'stroke-width="1.4" stroke-linecap="round" stroke-linejoin="round"/>'
    )

    stack = []
    for i, c in enumerate(structure):
        if c == "(":
            stack.append(i)
        elif c == ")":
            j = stack.pop()
            parts.append(
                f'<line x1="{px[i]:.1f}" y1="{py[i]:.1f}" x2="{px[j]:.1f}" y2="{py[j]:.1f}" '
                'stroke="var(--struct-pair)" stroke-width="1" opacity="0.55"/>'
            )

    highlight_kozak = set(range(kozak_index, kozak_index + len(gate.KOZAK_EUKARYOTIC)))
    highlight_aug = set(range(aug_index, aug_index + 3))
    highlight_binding = set(range(binding_start, binding_end))
    for i in range(len(seq)):
        if i in highlight_kozak:
            fill, r = "var(--struct-kozak)", 4.4
        elif i in highlight_aug:
            fill, r = "var(--struct-aug)", 4.4
        else:
            fill, r = "var(--struct-base)", 2.4
        if i in highlight_kozak or i in highlight_aug:
            ring = ' stroke="var(--struct-ring)" stroke-width="1.2"'
        elif i in highlight_binding:
            # Toehold/trigger-binding site: a coloured ring on the otherwise-neutral
            # base, not a fill change — Kozak/AUG are "what gets translated", this is
            # "what the trigger physically binds", a different kind of marker.
            fill, r = "var(--struct-base)", 3.0
            ring = ' stroke="var(--struct-binding)" stroke-width="1.4"'
        else:
            ring = ""
        parts.append(f'<circle cx="{px[i]:.1f}" cy="{py[i]:.1f}" r="{r}" fill="{fill}"{ring}/>')

    parts.append(
        f'<text x="{px[0]:.1f}" y="{py[0] - 8:.1f}" class="struct-end" '
        'text-anchor="middle">5&#8242;</text>'
    )
    parts.append(
        f'<text x="{px[-1]:.1f}" y="{py[-1] - 8:.1f}" class="struct-end" '
        'text-anchor="middle">3&#8242;</text>'
    )
    parts.append("</svg>")
    return "\n".join(parts)


def highlight_seq_html(seq, kozak_index, aug_index, binding_start, binding_end):
    parts = []
    cursor = 0
    spans = sorted(
        [
            (binding_start, binding_end, "binding"),
            (kozak_index, kozak_index + 6, "kozak"),
            (aug_index, aug_index + 3, "aug"),
        ]
    )
    for start, end, cls in spans:
        parts.append(_html.escape(seq[cursor:start]))
        parts.append(f'<span class="{cls}">{_html.escape(seq[start:end])}</span>')
        cursor = end
    parts.append(_html.escape(seq[cursor:]))
    return "".join(parts)

### Base-pair probability matrices, and the trigger-binding site

Two heatmaps per design — OFF (switch alone) and ON (switch + trigger, joined with `&`, a true dimer fold per `CLAUDE.md` §6's warning against concatenation — both via `FoldEngine.base_pair_probabilities`. The toehold (the region the trigger actually binds, per `ToeholdGate.generate_designs`: the first `toehold_length` nt after the leader) is outlined on both, and the boundary between switch and trigger is marked on the ON heatmap. Same switch-only truncation as the structure diagram, for the same reason.

This is a real diagnostic, not decoration: if the toehold region shows little or no new pairing with the trigger's columns/rows in the ON heatmap versus the OFF one, the mechanism this design is supposed to use — the trigger opening the switch — isn't actually what's happening in the fold prediction, whatever the summary metrics say.

In [ ]:
import base64
from io import BytesIO

import numpy as np
import matplotlib.pyplot as plt


def render_heatmap_png(matrix, title, binding_span=None, boundary=None):
    """Base-pair probability matrix as a base64 PNG. `binding_span` outlines the
    toehold (trigger-binding site); `boundary` (ON matrices only) marks where the
    switch ends and the trigger sequence begins."""
    arr = np.array(matrix)
    fig, ax = plt.subplots(figsize=(3.4, 3.0), dpi=130)
    im = ax.imshow(arr, cmap="Greens", vmin=0, vmax=1, origin="upper")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.tick_params(labelsize=6)
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=6)
    cbar.set_label("P(paired)", fontsize=7)

    if binding_span:
        start, end = binding_span
        rect = plt.Rectangle(
            (start - 0.5, start - 0.5), end - start, end - start,
            fill=False, edgecolor="#4C6E8C", linewidth=1.5, zorder=5,
        )
        ax.add_patch(rect)

    if boundary is not None:
        ax.axhline(boundary - 0.5, color="#B5760E", linewidth=1.1, linestyle="--")
        ax.axvline(boundary - 0.5, color="#B5760E", linewidth=1.1, linestyle="--")

    plt.tight_layout()
    buf = BytesIO()
    plt.savefig(buf, format="png", facecolor="white")
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("ascii")


def design_matrices(design):
    """(off_matrix, on_matrix, switch_only, trigger_sequence) for one design — same
    switch-only truncation as make_svg, for the same reason (payload head is real
    for evaluate_design's own metrics, not drawn here)."""
    aug_index = design.architecture["aug_index"]
    switch_only = design.sequence[: aug_index + 3]
    trigger_sequence = design.trigger_set.activators[0].sequence
    off_matrix = gate.folder.base_pair_probabilities(switch_only)
    on_matrix = gate.folder.base_pair_probabilities(f"{switch_only}&{trigger_sequence}")
    return off_matrix, on_matrix, switch_only, trigger_sequence


# Smoke-test on the top design before generating all 15 in the report below.
_off, _on, _switch, _trigger = design_matrices(report[0][0])
print(f"OFF matrix: {len(_off)}x{len(_off)}   ON matrix: {len(_on)}x{len(_on)}")
print(f"switch_only: {len(_switch)} nt   trigger: {len(_trigger)} nt")

In [ ]:
NEW_CSS = '''
  :root {
    --ground: #F6F8F4; --surface: #FFFFFF; --surface-2: #EEF2EA;
    --ink: #131A16; --ink-2: #46524B; --ink-3: #75837B; --line: #DCE4D8;
    --accent: #2E9E56; --accent-glow: #E4F7EA; --accent-ink: #146134;
    --structural: #4C6E8C; --structural-bg: #E9EFF4;
    --amber: #B5760E; --amber-bg: #FBF0DD;
    --shadow: 0 1px 2px rgba(19,26,22,0.04), 0 8px 24px -12px rgba(19,26,22,0.12);
    --struct-backbone: #C7D0CA; --struct-pair: #9FB0A7; --struct-base: #B7C4BC;
    --struct-kozak: #2E9E56; --struct-aug: #B5760E; --struct-ring: #FFFFFF;
    --struct-binding: #4C6E8C;
    --pill-utr-bg: #FBF0DD; --pill-utr-ink: #8A5A0A;
    --pill-cds-bg: #E9EFF4; --pill-cds-ink: #3E5A73;
  }
  @media (prefers-color-scheme: dark) {
    :root:not([data-theme="light"]) {
      --ground: #0E1512; --surface: #141D18; --surface-2: #1A2620;
      --ink: #E9F2EC; --ink-2: #AEC0B6; --ink-3: #7C8D83; --line: #26362D;
      --accent: #4CE07A; --accent-glow: #163524; --accent-ink: #7DF0A2;
      --structural: #8FB3D6; --structural-bg: #1B2A38;
      --amber: #E0AA4C; --amber-bg: #362B14;
      --shadow: 0 1px 2px rgba(0,0,0,0.3), 0 8px 24px -12px rgba(0,0,0,0.5);
      --struct-backbone: #3A4A41; --struct-pair: #5A6B60; --struct-base: #8A9990;
      --struct-kozak: #4CE07A; --struct-aug: #E0AA4C; --struct-ring: #0E1512;
      --struct-binding: #8FB3D6;
      --pill-utr-bg: #362B14; --pill-utr-ink: #E0AA4C;
      --pill-cds-bg: #1B2A38; --pill-cds-ink: #8FB3D6;
    }
  }
  :root[data-theme="dark"] {
    --ground: #0E1512; --surface: #141D18; --surface-2: #1A2620;
    --ink: #E9F2EC; --ink-2: #AEC0B6; --ink-3: #7C8D83; --line: #26362D;
    --accent: #4CE07A; --accent-glow: #163524; --accent-ink: #7DF0A2;
    --structural: #8FB3D6; --structural-bg: #1B2A38;
    --amber: #E0AA4C; --amber-bg: #362B14;
    --shadow: 0 1px 2px rgba(0,0,0,0.3), 0 8px 24px -12px rgba(0,0,0,0.5);
    --struct-backbone: #3A4A41; --struct-pair: #5A6B60; --struct-base: #8A9990;
    --struct-kozak: #4CE07A; --struct-aug: #E0AA4C; --struct-ring: #0E1512;
    --struct-binding: #8FB3D6;
    --pill-utr-bg: #362B14; --pill-utr-ink: #E0AA4C;
    --pill-cds-bg: #1B2A38; --pill-cds-ink: #8FB3D6;
  }
  * { box-sizing: border-box; }
  body { margin: 0; background: var(--ground); color: var(--ink); font-family: "IBM Plex Sans", system-ui, sans-serif; padding: 0 20px; }
  .mono { font-family: "IBM Plex Mono", ui-monospace, monospace; font-variant-numeric: tabular-nums; }
  .wrap { max-width: 1040px; margin: 0 auto; padding-block: 48px 80px; }
  header.report { margin-bottom: 40px; }
  .eyebrow { font-family: "IBM Plex Mono", monospace; font-size: 12px; letter-spacing: 0.08em; text-transform: uppercase; color: var(--structural); margin: 0 0 10px; }
  h1 { font-size: clamp(28px, 4vw, 38px); font-weight: 700; line-height: 1.15; letter-spacing: -0.01em; margin: 0 0 12px; text-wrap: balance; }
  .dek { font-size: 16px; line-height: 1.6; color: var(--ink-2); max-width: 68ch; margin: 0 0 28px; }
  .meta-strip { display: grid; grid-template-columns: repeat(auto-fit, minmax(140px, 1fr)); gap: 1px; background: var(--line); border: 1px solid var(--line); border-radius: 10px; overflow: hidden; }
  .meta-item { background: var(--surface); padding: 14px 16px; }
  .meta-item .k { font-size: 11px; letter-spacing: 0.06em; text-transform: uppercase; color: var(--ink-3); margin: 0 0 4px; }
  .meta-item .v { font-size: 14px; font-weight: 600; color: var(--ink); }
  section { margin-bottom: 48px; }
  h2 { font-size: 13px; font-weight: 600; letter-spacing: 0.06em; text-transform: uppercase; color: var(--ink-3); margin: 0 0 4px; }
  .section-title { font-size: 22px; font-weight: 700; letter-spacing: -0.01em; color: var(--ink); margin: 0 0 6px; }
  .section-note { font-size: 14px; color: var(--ink-2); line-height: 1.6; margin: 0 0 20px; max-width: 70ch; }
  .pill { display: inline-block; font-family: "IBM Plex Mono", monospace; font-size: 10.5px; font-weight: 600; letter-spacing: 0.03em; text-transform: uppercase; padding: 2px 7px; border-radius: 999px; }
  .pill-utr { background: var(--pill-utr-bg); color: var(--pill-utr-ink); }
  .pill-cds { background: var(--pill-cds-bg); color: var(--pill-cds-ink); }
  .glossary { background: var(--surface); border: 1px solid var(--line); border-radius: 12px; padding: 6px 22px; box-shadow: var(--shadow); }
  .glossary dl { display: grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap: 4px 24px; }
  .glossary .term { padding: 14px 0; border-bottom: 1px solid var(--line); }
  .glossary dt { font-family: "IBM Plex Mono", monospace; font-size: 12.5px; font-weight: 600; color: var(--accent-ink); margin: 0 0 3px; }
  .glossary dd { font-size: 13px; color: var(--ink-2); line-height: 1.5; margin: 0; }
  .chart { background: var(--surface); border: 1px solid var(--line); border-radius: 12px; padding: 20px 20px 16px; box-shadow: var(--shadow); }
  .bar-row { display: grid; grid-template-columns: 28px 170px 1fr; align-items: center; gap: 12px; padding: 7px 0; border-bottom: 1px solid var(--line); }
  .bar-row:last-child { border-bottom: none; }
  .bar-rank { font-family: "IBM Plex Mono", monospace; font-size: 12px; color: var(--ink-3); }
  .bar-label { font-family: "IBM Plex Mono", monospace; font-size: 12.5px; color: var(--ink-2); white-space: nowrap; overflow: hidden; text-overflow: ellipsis; display: flex; align-items: center; gap: 6px; }
  .bar-track { position: relative; height: 18px; background: var(--surface-2); border-radius: 4px; overflow: hidden; }
  .bar-fill { height: 100%; background: var(--accent); border-radius: 4px; }
  .bar-value { position: absolute; right: 8px; top: 50%; transform: translateY(-50%); font-family: "IBM Plex Mono", monospace; font-size: 11.5px; font-weight: 600; color: var(--ink); }
  .table-wrap { overflow-x: auto; border: 1px solid var(--line); border-radius: 12px; background: var(--surface); box-shadow: var(--shadow); }
  table { width: 100%; border-collapse: collapse; font-size: 13px; min-width: 720px; }
  th { text-align: left; font-size: 11px; letter-spacing: 0.05em; text-transform: uppercase; color: var(--ink-3); font-weight: 600; padding: 12px 14px; border-bottom: 1px solid var(--line); background: var(--surface-2); }
  td { padding: 10px 14px; border-bottom: 1px solid var(--line); color: var(--ink-2); }
  tr:last-child td { border-bottom: none; }
  td.num { text-align: right; }
  .accent-text { color: var(--accent-ink); font-weight: 600; }
  .cards { display: flex; flex-direction: column; gap: 14px; }
  .card { background: var(--surface); border: 1px solid var(--line); border-radius: 12px; padding: 18px 20px; box-shadow: var(--shadow); }
  .card-head { display: flex; align-items: flex-start; gap: 14px; margin-bottom: 14px; }
  .card-rank { font-family: "IBM Plex Mono", monospace; font-size: 13px; font-weight: 600; color: var(--accent-ink); background: var(--accent-glow); border-radius: 6px; padding: 3px 8px; flex-shrink: 0; }
  .card-id { flex: 1; min-width: 0; }
  .card-id h3 { font-family: "IBM Plex Mono", monospace; font-size: 13.5px; font-weight: 600; color: var(--ink); margin: 0 0 3px; word-break: break-all; }
  .card-sub { font-size: 12.5px; color: var(--ink-3); margin: 0; display: flex; align-items: center; gap: 8px; flex-wrap: wrap; }
  .card-len { font-family: "IBM Plex Mono", monospace; font-size: 18px; font-weight: 600; color: var(--structural); flex-shrink: 0; text-align: right; }
  .card-len span { display: block; font-size: 10px; color: var(--ink-3); font-weight: 500; }
  .card-body { display: grid; grid-template-columns: 260px 1fr; gap: 18px; }
  .card-structure { display: flex; flex-direction: column; align-items: center; background: var(--surface-2); border-radius: 8px; padding: 6px; }
  .structure-svg { width: 100%; height: auto; }
  .struct-caption { font-size: 10.5px; color: var(--ink-3); text-align: center; margin: 0 4px 6px; line-height: 1.4; }
  .struct-end { font-family: "IBM Plex Mono", monospace; font-size: 8px; fill: var(--ink-3); }
  .param-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(100px, 1fr)); gap: 10px 16px; margin: 0 0 16px; padding: 12px 14px; background: var(--surface-2); border-radius: 8px; }
  .param-grid dt { font-size: 10.5px; letter-spacing: 0.03em; text-transform: uppercase; color: var(--ink-3); margin: 0 0 2px; }
  .param-grid dd { font-size: 13.5px; font-weight: 600; color: var(--ink); margin: 0; }
  .seq-label { font-size: 10.5px; letter-spacing: 0.05em; text-transform: uppercase; color: var(--ink-3); margin: 0 0 6px; }
  .seq { font-size: 12px; line-height: 1.9; word-break: break-all; background: var(--surface-2); border-radius: 8px; padding: 10px 12px; color: var(--ink-2); }
  .seq .kozak { background: var(--accent-glow); color: var(--accent-ink); font-weight: 600; border-radius: 3px; }
  .seq .aug { background: var(--amber-bg); color: var(--amber); font-weight: 700; border-radius: 3px; }
  .seq .binding { text-decoration: underline; text-decoration-color: var(--struct-binding); text-decoration-thickness: 2px; text-underline-offset: 2px; }
  .seq-legend { display: flex; align-items: center; gap: 6px; font-size: 11.5px; color: var(--ink-3); margin-top: 8px; flex-wrap: wrap; }
  .seq-legend .dot { width: 9px; height: 9px; border-radius: 2px; display: inline-block; }
  .kozak-dot { background: var(--accent); }
  .aug-dot { background: var(--amber); margin-left: 10px; }
  .binding-dot { border: 2px solid var(--struct-binding); background: transparent; margin-left: 10px; }
  .heatmap-row { display: flex; gap: 14px; margin-top: 14px; flex-wrap: wrap; }
  .heatmap-row figure { margin: 0; flex: 1; min-width: 220px; text-align: center; }
  .heatmap-row img { width: 100%; height: auto; border-radius: 6px; border: 1px solid var(--line); }
  .heatmap-row figcaption { font-size: 10.5px; color: var(--ink-3); margin-top: 4px; }
  .callout { background: var(--structural-bg); border: 1px solid var(--line); border-radius: 12px; padding: 20px 22px; font-size: 13.5px; line-height: 1.7; color: var(--ink-2); }
  .callout strong { color: var(--ink); }
  .callout ul { margin: 10px 0 0; padding-left: 20px; }
  .callout li { margin-bottom: 6px; }
  footer { border-top: 1px solid var(--line); padding-top: 20px; font-size: 12px; color: var(--ink-3); }
  @media (max-width: 680px) {
    .card-body { grid-template-columns: 1fr; }
    .card-structure { max-width: 260px; margin: 0 auto; }
    .bar-row { grid-template-columns: 22px 1fr; }
    .bar-label { grid-column: 2; }
    .bar-track { grid-column: 1 / -1; }
  }
'''

GLOSSARY_TERMS = [
    ("Trigger", "The mRNA window this switch responds to — a 30 or 36 nt segment of the AREG transcript, picked by TriggerScorer."),
    ("Region (CDS / UTR)", "Where the trigger's window starts on the transcript: inside/before the coding sequence, or in the 3' untranslated region (from 100 nt before the CDS ends to the transcript's end)."),
    ("Toehold", "Length of the single-stranded region the trigger physically binds first, in nucleotides — outlined in the structure diagram and underlined in the sequence below."),
    ("Loop", "Length of the hairpin's loop, in nucleotides — free to vary in this layout since Kozak no longer lives inside it."),
    ("Kozak spacer", "Optional gap, in nucleotides, between the closed hairpin and the Kozak element."),
    ("Payload head", "How many of the real effector gene's own leading nucleotides are folded in for evaluation (not shown in the structure diagram — see §6)."),
    ("Ensemble ΔG", "Free energy from ViennaRNA's partition function — accounts for every possible fold, not just one. More negative is more stable. kcal/mol."),
    ("MFE ΔG", "Free energy of the single lowest-energy predicted structure — the one drawn in the diagram. kcal/mol."),
    ("Leakage (OFF)", "How accessible the toehold+stem region is with no trigger present, 0-1. Lower is better — a leaky switch turns on by itself."),
    ("Dynamic range", "Ratio of trigger-bound to trigger-free accessibility at that same region. Higher is better; above 1.0 means the trigger genuinely opens the switch."),
    ("GC content", "Percent G+C in the full sequence (switch + payload head). Extremes hurt both synthesis and folding behaviour."),
]


def render_glossary_html():
    terms = "".join(
        f'<div class="term"><dt>{_html.escape(name)}</dt><dd>{_html.escape(desc)}</dd></div>'
        for name, desc in GLOSSARY_TERMS
    )
    return f"""
  <section>
    <h2>Glossary</h2>
    <p class="section-title">What each parameter below means</p>
    <div class="glossary"><dl>{terms}</dl></div>
  </section>"""


def render_report_html(report_pairs, gate, transcript_gene="AREG"):
    entries = []
    for rank, (d, m) in enumerate(report_pairs, 1):
        arch = d.architecture
        off_matrix, on_matrix, switch_only, _trig_seq = design_matrices(d)
        binding_span = (arch["leader_len"], arch["leader_len"] + arch["toehold_length"])
        entries.append(
            {
                "rank": rank,
                "design": d,
                "metrics": m,
                "region": "utr" if in_3utr(d) else "cds",
                "ensemble_energy": gate.folder.partition(d.sequence),
                "kozak_index": d.sequence.find(gate.KOZAK_EUKARYOTIC),
                "binding_span": binding_span,
                "off_heatmap": render_heatmap_png(off_matrix, "OFF (switch alone)", binding_span=binding_span),
                "on_heatmap": render_heatmap_png(
                    on_matrix, "ON (switch + trigger)", binding_span=binding_span, boundary=len(switch_only)
                ),
            }
        )

    max_dyn = max(e["metrics"]["dynamic_range"] for e in entries)

    bar_rows = []
    for e in entries:
        d, m = e["design"], e["metrics"]
        pct = round(m["dynamic_range"] / max_dyn * 100, 1)
        pill = f'<span class="pill pill-{e["region"]}">{e["region"]}</span>'
        bar_rows.append(
            f"""
        <div class="bar-row">
          <div class="bar-rank">{e["rank"]:02d}</div>
          <div class="bar-label">{_html.escape(d.trigger_set.activators[0].trigger_id)} {pill}</div>
          <div class="bar-track"><div class="bar-fill" style="width:{pct}%"></div><span class="bar-value">{m["dynamic_range"]:.3f}</span></div>
        </div>"""
        )

    table_rows = []
    for e in entries:
        d, m, arch = e["design"], e["metrics"], e["design"].architecture
        pill = f'<span class="pill pill-{e["region"]}">{e["region"]}</span>'
        table_rows.append(
            f"""
        <tr>
          <td class="num">{e["rank"]:02d}</td>
          <td class="mono">{_html.escape(d.trigger_set.activators[0].trigger_id)}</td>
          <td>{pill}</td>
          <td class="num mono">{arch["toehold_length"]}/{arch["loop_len"]}/{arch["kozak_linker_len"]}</td>
          <td class="num mono">{d.length}</td>
          <td class="num mono">{m["predicted_leakage"]:.3f}</td>
          <td class="num mono accent-text">{m["dynamic_range"]:.3f}</td>
          <td class="num mono">{e["ensemble_energy"]:.1f}</td>
          <td class="num mono">{m["gc_content"]:.1f}%</td>
        </tr>"""
        )

    cards = []
    for e in entries:
        d, m, arch = e["design"], e["metrics"], e["design"].architecture
        pill = f'<span class="pill pill-{e["region"]}">{e["region"].upper()}</span>'
        svg = make_svg(d)
        binding_start, binding_end = e["binding_span"]
        cards.append(
            f"""
      <article class="card">
        <div class="card-head">
          <div class="card-rank">{e["rank"]:02d}</div>
          <div class="card-id">
            <h3>{_html.escape(d.design_id)}</h3>
            <p class="card-sub">trigger {_html.escape(d.trigger_set.activators[0].trigger_id)} &middot; start {d.trigger_set.activators[0].start_index} nt {pill}</p>
          </div>
          <div class="card-len">{d.length}<span>nt</span></div>
        </div>
        <div class="card-body">
          <div class="card-structure">
            {svg}
            <p class="struct-caption">predicted fold (MFE) — switch only, payload head omitted for clarity</p>
          </div>
          <div class="card-data">
            <dl class="param-grid">
              <div><dt>Toehold</dt><dd class="mono">{arch["toehold_length"]} nt</dd></div>
              <div><dt>Loop</dt><dd class="mono">{arch["loop_len"]} nt</dd></div>
              <div><dt>Kozak spacer</dt><dd class="mono">{arch["kozak_linker_len"]} nt</dd></div>
              <div><dt>Payload head</dt><dd class="mono">{arch["payload_head_length"]} nt</dd></div>
              <div><dt>Ensemble &#916;G</dt><dd class="mono">{e["ensemble_energy"]:.1f} kcal/mol</dd></div>
              <div><dt>MFE &#916;G</dt><dd class="mono">{m["gate_folding_energy"]:.1f} kcal/mol</dd></div>
              <div><dt>Leakage (OFF)</dt><dd class="mono">{m["predicted_leakage"]:.3f}</dd></div>
              <div><dt>Dynamic range</dt><dd class="mono accent-text">{m["dynamic_range"]:.3f}</dd></div>
              <div><dt>GC content</dt><dd class="mono">{m["gc_content"]:.1f}%</dd></div>
            </dl>
            <div class="seq-block">
              <div class="seq-label">sequence (5&#8242;&rarr;3&#8242;)</div>
              <div class="seq mono">{highlight_seq_html(d.sequence, e["kozak_index"], arch["aug_index"], binding_start, binding_end)}</div>
              <div class="seq-legend"><span class="dot kozak-dot"></span>Kozak (GCCACC)<span class="dot aug-dot"></span>start codon (AUG)<span class="dot binding-dot"></span>toehold / trigger-binding site</div>
            </div>
          </div>
        </div>
        <div class="heatmap-row">
          <figure><img src="data:image/png;base64,{e["off_heatmap"]}" alt="OFF-state base-pair probability matrix"><figcaption>OFF — switch alone</figcaption></figure>
          <figure><img src="data:image/png;base64,{e["on_heatmap"]}" alt="ON-state base-pair probability matrix"><figcaption>ON — switch + trigger (dashed line = boundary between the two)</figcaption></figure>
        </div>
      </article>"""
        )

    return f"""<title>AREG Toehold Candidates</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:wght@400;500;600;700&family=IBM+Plex+Mono:wght@400;500;600&display=swap">
<style>{NEW_CSS}</style>
<div class="wrap">
  <header class="report">
    <p class="eyebrow">Switch design report &middot; generated from this notebook</p>
    <h1>AREG toehold candidates &mdash; trailing-Kozak, GFP payload</h1>
    <p class="dek">
      Top {len(entries)} of {len(designs)} candidate switches, pooled from {len(top_triggers)}
      trigger candidates against a full-length human <em>{transcript_gene}</em> mRNA (at
      least {MIN_UTR_IN_REPORT} guaranteed from the 3&#8242; UTR), ranked by
      <code class="mono">dynamic_range</code>.
    </p>
    <div class="meta-strip">
      <div class="meta-item"><div class="k">Host</div><div class="v">{gate.host.value.title()}</div></div>
      <div class="meta-item"><div class="k">Gate version</div><div class="v">{gate.version}</div></div>
      <div class="meta-item"><div class="k">Layout</div><div class="v">Trailing Kozak</div></div>
      <div class="meta-item"><div class="k">Trigger pool</div><div class="v">{len(top_triggers)} of {len(candidates)}</div></div>
      <div class="meta-item"><div class="k">3&#8242; UTR triggers</div><div class="v">{n_utr_selected} of {len(top_triggers)}</div></div>
      <div class="meta-item"><div class="k">Effector payload</div><div class="v">GFP-family, {len(gate.payload)} nt</div></div>
    </div>
  </header>
  {render_glossary_html()}
  <section>
    <h2>Ranking</h2>
    <p class="section-title">Dynamic range across the top {len(entries)}</p>
    <div class="chart">{"".join(bar_rows)}</div>
  </section>
  <section>
    <h2>Summary</h2>
    <p class="section-title">All {len(entries)} candidates, at a glance</p>
    <p class="section-note">Ensemble &#916;G is the partition-function free energy
    (<code class="mono">FoldEngine.partition</code>), not MFE.</p>
    <div class="table-wrap">
      <table>
        <thead><tr><th>#</th><th>Trigger</th><th>Region</th><th>Geometry (nt)</th><th>Length</th><th>Leakage</th><th>Dyn. range</th><th>Ensemble &#916;G</th><th>GC</th></tr></thead>
        <tbody>{"".join(table_rows)}</tbody>
      </table>
    </div>
  </section>
  <section>
    <h2>Detail</h2>
    <p class="section-title">Every candidate, with predicted structure and base-pair probabilities</p>
    <p class="section-note">MFE fold via the same injected <code class="mono">FoldEngine</code>
    used throughout this notebook — switch only, payload head omitted so Kozak/AUG
    (outlined) land at the 3&#8242; end. The toehold is outlined in blue in both the
    structure diagram and the heatmaps below it — that's the trigger's actual binding
    site, not just "somewhere in the switch."</p>
    <div class="cards">{"".join(cards)}</div>
  </section>
  <section>
    <div class="callout">
      <strong>Reading this report.</strong> Raw, unranked-by-the-real-pipeline
      candidates — <code class="mono">engine.scoring</code> has not run on them.
      <ul>
        <li>The three 3&#8242; UTR designs are guaranteed representation, not
          guaranteed performance.</li>
        <li>Only the "trailing" layout and {len(top_triggers)} (of {len(candidates)})
          trigger candidates were explored.</li>
        <li>The leakage/dynamic-range proxy for "trailing" is new, unreviewed
          science.</li>
        <li>The 3&#8242; UTR boundary is a longest-ORF heuristic, not real gene
          annotation.</li>
        <li>The base-pair probability matrices are computed for the switch alone
          (OFF) and switch+trigger (ON), both without the payload head — the same
          simplification as the structure diagrams, for the same reason.</li>
      </ul>
    </div>
  </section>
  <footer>Generated from
  <code class="mono">src/engine/gates/notebooks/toehold/eukaryotic-usage-example.ipynb</code>
  &middot; <code class="mono">ToeholdGate</code> v{gate.version} &middot; real
  <code class="mono">FoldEngine</code> (ViennaRNA), no stub folding.</footer>
</div>
"""

In [ ]:
from pathlib import Path

out_path = Path.cwd()
for _base in (out_path, *out_path.parents):
    if (_base / "pyproject.toml").exists():
        out_path = _base / "var" / "reports" / "areg_toehold_report.html"
        break

out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(render_report_html(report, gate))

print(f"wrote {out_path.stat().st_size:,} bytes to {out_path}")
print("\nTo publish: hand this file path to Claude's Artifact tool (outside this")
print("notebook), or open it directly in a browser to view/print it locally.")

## Where this fits in a real run

This notebook now pools 904 designs across 150 triggers (41 from the 3&#8242; UTR, 109
from the CDS region) and one layout — closer to a real run, but still a slice of it.
In the actual pipeline:

- Every trigger `TriggerScorer` keeps (not just the top `N_TRIGGERS`) gets checked
  with `is_compatible` and, if it passes, run through `generate_designs` — and by
  default that sweeps **both** `"loop"` and `"trailing"` layouts (this notebook
  restricted to `"trailing"` only via `kozak_layouts` — see cell 4). No 3&#8242; UTR
  quota exists anywhere in `engine.stages.triggers` — the floor enforced here (cell 7)
  is a notebook-level policy, not an engine feature; if a real run needs this
  guarantee, it belongs in `Constraints` or `TriggerScorer` itself, not reimplemented
  per caller.
- Every design's raw metrics go through `engine.scoring` — `build_metrics`,
  `weighted_score`, `failed_filter`, `rank_candidates` — which is what actually
  decides which designs survive and how they rank, comparably with every other gate
  family's designs, every trigger, **and across layouts**. Neither this notebook nor
  the gate itself picks a winner — sorting by `dynamic_range` alone (cell 17) is a
  stand-in for that, not the real ranking.
- `CandidateStore` records provenance and writes the stage snapshot; nothing here
  hand-rolls a results CSV.
- The **payload** folds into evaluation when known (`PAYLOAD_CDS` in cell 4) but the
  *actual* fused construct is still assembled later, at plasmid assembly
  (`PlasmidBuilder.build(circuit, DesiredOutcome.CUSTOM, custom_payload=...)`), using
  the complete gene, not just its folded-in head.

The two-input AND version (`EukaryoticToeholdAndGate`) is **not** implemented yet —
its `generate_designs` still raises `NotImplementedError("Step 5")`. This notebook
only covers the single-input case.

Open questions this work surfaced, not resolved here (see commits `d754812`,
`ee2f5f6`, `74bf7b4`): whether `"loop"` should still ship as the eukaryotic default
now that `"trailing"` exists (and now that `"loop"`'s Kozak-AUG adjacency is known to
be broken); whether `TRAILING_LOOP_LENGTHS`/`KOZAK_LINKER_LENGTHS` are the right
ranges to sweep; whether the `"trailing"` leakage proxy (toehold+stem accessibility)
is the right measurement for scanning-ribosome blockage at all; whether fusing the
payload directly after the switch's own placeholder AUG (rather than dropping that AUG
in favour of the payload's own) is the right call; the CDS boundary here is a
longest-ORF heuristic, not real annotation — wrong for any transcript with a shorter
true CDS or a non-canonical start; and whether a 3&#8242; UTR floor belongs in
`Constraints` as a first-class, engine-level concept rather than a per-notebook
policy, given the trigger scoring map's own emphasis on where a trigger sits along the
transcript.